# Structured Output

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI

# Structured Output에서는 일관된 결과를 위해 temperature=0이 적합하다
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

---

## Structured Output

- LLM의 응답을 정해진 구조(스키마)로 받는 기능
- LLM의 응답은 정형화되어있지 않은 텍스트, 코드에서 활용하기 어려우며, Structured Output은 이 문제를 해결한다
- 자유로운 텍스트 대신 Python 객체로 받을 수 있다
- Pydantic 모델로 스키마를 정의한다

```python
# ❌ 텍스트 응답 — 파싱이 어렵다
# "감정: 부정, 강도: 강함, 근거: '최악'이라는 표현"

# ✅ 구조화된 응답 — 바로 코드에서 사용 가능
# SentimentResult(sentiment='부정', intensity='강함', reason="'최악'이라는 표현")
```

### with_structured_output()

`with_structured_output()`을 사용하면 LLM이 자유 텍스트 대신 지정한 Pydantic 모델 형태로 응답한다. 내부적으로는 OpenAI의 function calling 기능을 활용하여 JSON을 생성하고, 이를 Pydantic 모델로 변환한다.

`Field(description=...)`은 LLM에게 "이 필드에 무엇을 넣어야 하는지" 설명해주는 역할이다. description이 명확할수록 LLM이 정확한 값을 반환한다.

`with_structured_output()`은 내부적으로 function calling을 사용하여 API 레벨에서 스키마를 전달하므로, **별도의 프롬프트(format_instructions)나 파서(OutputParser)가 필요 없다.** 기존의 `prompt | llm | parser` 체인 대신 `structured_llm.invoke()`만으로 Pydantic 객체를 바로 받을 수 있다. 단, 시스템 역할이나 추가 지시가 필요한 경우에는 `prompt | structured_llm` 형태로 프롬프트만 추가하면 된다.

In [6]:
from pydantic import BaseModel, Field

# 출력 스키마 정의
class SentimentResult(BaseModel):
    sentiment: str = Field(description="감정 (긍정/부정/중립)")
    intensity: str = Field(description="강도 (강함/보통/약함)")
    reason: str = Field(description="판단 근거")

structured_llm = llm.with_structured_output(SentimentResult)

result = structured_llm.invoke("이 제품 정말 최악이에요. 다시는 안 살 겁니다.")

print(result)
print(f"\n감정: {result.sentiment}")
print(f"강도: {result.intensity}")
print(f"근거: {result.reason}")

sentiment='부정' intensity='강함' reason='제품에 대한 불만이 매우 강하게 표현되고 있으며, 재구매 의사가 전혀 없다는 점에서 부정적인 감정이 뚜렷하게 드러납니다.'

감정: 부정
강도: 강함
근거: 제품에 대한 불만이 매우 강하게 표현되고 있으며, 재구매 의사가 전혀 없다는 점에서 부정적인 감정이 뚜렷하게 드러납니다.


### 체인에서 Structured Output 사용하기

In [3]:
from langchain_core.prompts import ChatPromptTemplate

In [7]:
class MovieReview(BaseModel):
    title: str = Field(description="영화 제목")
    rating: int = Field(description="평점 (1-10)")
    pros: list[str] = Field(description="장점 목록")
    cons: list[str] = Field(description="단점 목록")
    summary: str = Field(description="한줄 요약")

prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 영화 평론가야. 주어진 영화 리뷰를 분석해줘."),
    ("human", "{review}"),
])

structured_llm = llm.with_structured_output(MovieReview)
chain = prompt | structured_llm

result = chain.invoke({
    "review": "인터스텔라는 정말 대단한 영화였습니다. 시각적으로 압도적이고 음악도 훌륭했어요. 다만 러닝타임이 너무 길고 중반부가 조금 지루했습니다."
})

print(f"제목: {result.title}")
print(f"평점: {result.rating}/10")
print(f"장점: {result.pros}")
print(f"단점: {result.cons}")
print(f"요약: {result.summary}")

제목: 인터스텔라
평점: 8/10
장점: ['시각적으로 압도적이다', '음악이 훌륭하다']
단점: ['러닝타임이 너무 길다', '중반부가 지루하다']
요약: 인터스텔라는 시각적 아름다움과 뛰어난 음악으로 관객을 사로잡는 대단한 영화입니다.


### Enum을 활용한 복잡한 스키마

**Enum(열거형)** 은 미리 정해진 값들만 허용하는 타입이다. `str` 대신 Enum을 쓰면 LLM이 임의의 문자열을 반환하는 것을 방지할 수 있다.

```python
from enum import Enum

# str을 상속하면 값이 문자열로 직렬화된다
class Color(str, Enum):
    RED = "red"
    GREEN = "green"
    BLUE = "blue"

print(Color.RED)        # Color.RED
print(Color.RED.value)  # "red"

# ❌ 정의되지 않은 값은 에러
# Color("yellow")  → ValueError
```

- `str` 필드: LLM이 `"긴급"`, `"높음"`, `"상"` 등 자유롭게 반환 → 일관성 없음
- `Enum` 필드: `"high"`, `"medium"`, `"low"` 중 하나만 반환 → 코드에서 안전하게 분기 가능

Pydantic 모델에서 `category: Category`처럼 타입을 Enum으로 지정하면, LLM의 function calling 스키마에 허용 값 목록이 전달되어 정해진 값만 반환하도록 강제된다.

In [12]:
from enum import Enum

class Category(str, Enum):
    BUG = "bug"
    FEATURE = "feature"
    QUESTION = "question"
    DOCS = "docs"

class Priority(str, Enum):
    HIGH = "high"
    MEDIUM = "medium"
    LOW = "low"

class IssueAnalysis(BaseModel):
    category: Category = Field(description="이슈 카테고리")
    priority: Priority = Field(description="우선순위")
    title: str = Field(description="이슈 제목 (간결하게)")
    description: str = Field(description="이슈 설명")
    affected_components: list[str] = Field(description="영향받는 컴포넌트 목록")

structured_llm = llm.with_structured_output(IssueAnalysis)

result = structured_llm.invoke(
    "로그인 페이지에서 비밀번호를 입력하고 엔터를 누르면 화면이 깜빡이면서 입력값이 초기화됩니다. 크롬 브라우저에서만 발생하고, 사파리에서는 정상입니다."
)

print(f"카테고리: {result.category.value}")
print(f"우선순위: {result.priority.value}")
print(f"제목: {result.title}")
print(f"설명: {result.description}")
print(f"영향 컴포넌트: {result.affected_components}")

카테고리: bug
우선순위: high
제목: 로그인 페이지 비밀번호 입력 후 엔터 시 화면 깜빡임 및 초기화 문제
설명: 로그인 페이지에서 비밀번호를 입력하고 엔터를 누르면 화면이 깜빡이면서 입력값이 초기화되는 문제가 발생합니다. 이 문제는 크롬 브라우저에서만 발생하며, 사파리에서는 정상적으로 작동합니다.
영향 컴포넌트: ['로그인 페이지', '비밀번호 입력 필드', '크롬 브라우저']


---

## Structured Output 실패 처리

`with_structured_output()`은 function calling 기반이라 스키마 준수율이 매우 높지만, 드물게 실패할 수 있다. 이때는 `.with_retry()`로 체인을 재시도할 수 있다.

```python
# 실패 시 최대 3번 자동 재시도
chain = prompt | structured_llm
safe_chain = chain.with_retry(stop_after_attempt=3)
```

### max_retries vs with_retry

이 둘은 재시도하는 **위치**가 다르다.

```
[LangChain 체인] → [LLM API 요청] → 네트워크/서버 → [응답 수신] → [파싱/검증]
                    ↑ max_retries                       ↑ with_retry
```

| | `max_retries` | `.with_retry()` |
|---|---|---|
| **설정 위치** | `ChatOpenAI(max_retries=3)` | `chain.with_retry(stop_after_attempt=3)` |
| **재시도 주체** | OpenAI SDK (HTTP 클라이언트) | LangChain (체인) |
| **대상 에러** | 네트워크 에러, 타임아웃, 429 Rate Limit | 파싱 실패, 스키마 불일치 |
| **동작** | 같은 HTTP 요청을 다시 보냄 | 체인 전체를 처음부터 다시 실행 (= LLM 재호출) |

- `max_retries`: **서버가 응답을 안 줄 때** — OpenAI SDK가 같은 요청을 재전송
- `with_retry`: **응답은 왔는데 내용이 잘못됐을 때** — LangChain이 체인을 처음부터 다시 실행하므로 LLM API를 다시 호출한다

---

## 실습 1: 이력서 분석기

이력서 텍스트를 입력하면 스킬, 경력 연차, 적합 포지션, 강점/약점을 구조화하여 반환하는 체인을 만들어보자.

### 요구사항

1. `Level` Enum을 정의하여 포지션 레벨을 `junior`, `mid`, `senior` 중 하나로 제한할 것
2. `ResumeAnalysis` Pydantic 모델을 정의할 것 (이름, 기술 스택, 경력 연차, 레벨, 강점, 약점, 한줄 요약)
3. `prompt | structured_llm` 체인으로 구성할 것

In [17]:
# 예시 이력서 1
resume1 = """
김영희 | 백엔드 개발자

경력:
- ABC 스타트업 (2022 ~ 2025) - 백엔드 개발자
  - FastAPI 기반 REST API 설계 및 개발 (사내 ERP 시스템)
  - PostgreSQL 스키마 설계, 쿼리 최적화 (슬로우 쿼리 80% 개선)
  - Redis 캐싱 레이어 도입으로 API 응답 속도 40% 향상
  - Docker + GitHub Actions 기반 CI/CD 파이프라인 구축
  - AWS EC2/S3/RDS 인프라 운영

- DEF 솔루션즈 (2020 ~ 2022) - 주니어 개발자
  - FastAPI 기반 사내 인사관리 시스템 개발 및 유지보수
  - MySQL → PostgreSQL 마이그레이션 수행
  - Celery를 활용한 비동기 메일 발송 시스템 구현
  - REST API 문서화 (Swagger)

기술 스택: Python, FastAPI, PostgreSQL, MySQL, Redis, Docker, AWS, Git
학력: 컴퓨터공학 학사 (2020 졸업)
자격증: 정보처리기사
""".strip()

# 예시 이력서 2
resume2 = """
박지훈 | 프론트엔드 개발자

경력:
- GHI 커머스 (2024 ~ 2025) - 프론트엔드 개발자
  - React + TypeScript 기반 이커머스 웹앱 개발
  - 공통 UI 컴포넌트 라이브러리 구축 (Button, Modal, Input 등)
  - React Query 도입으로 서버 상태 관리 개선

학력: 컴퓨터공학 학사 (2024 졸업)
부트캠프: 프론트엔드 개발 과정 수료 (2023)
기술 스택: JavaScript, TypeScript, React, Next.js, Tailwind CSS, Git
""".strip()

### 예시 응답

```
이름: 김영희
기술: ['Python', 'FastAPI', 'PostgreSQL', 'MySQL', 'Redis', 'Docker', 'AWS', 'Git']
경력: 5년
레벨: mid
강점: ['FastAPI 기반 API 설계 및 개발 경험이 풍부', 'DB 최적화 및 마이그레이션 경험', 'CI/CD 및 클라우드 인프라 운영 가능']
약점: ['프론트엔드 경험 부재', '대규모 트래픽 처리 경험 미확인', '팀 리딩 경험 미확인']
요약: Python 백엔드 5년차 개발자로 API 설계, DB 최적화, 인프라 운영까지 폭넓은 실무 경험 보유
```

In [26]:
# 여기에 코드를 작성하세요

class Level(str, Enum):
    junior = "junior"
    mid = "mid"
    senior = "senior"
    

class Review(BaseModel):
    name: str = Field(description="이름")
    stack: list[str] = Field(description="기술을 한눈에 정리")
    경력: str = Field(description="총경력 몇년인지")
    레벨: Level = Field(description="레벨이 어디인지")
    강점: list[str] = Field(description="강점 정리 ")
    약점: list[str] = Field(description="약점 정리")
    요약: str = Field(description="이력서 한줄요약")


prompt = ChatPromptTemplate.from_messages([
    ("system", "분석가야 주어진 이력서를 분석해줘."),
    ("human", "{text}"),
])

structured_llm = llm.with_structured_output(Review)
chain = prompt | structured_llm 

result = chain.invoke({"text" :resume1})

print(f"이름: {result.name}")
print(f"기술: {result.stack}")
print(f"경력: {result.경력}")
print(f"레벨: {result.레벨.value}")
print(f"강점: {result.강점}")
print(f"약점: {result.약점}")
print(f"요약: {result.요약}")

이름: 김영희
기술: ['Python', 'FastAPI', 'PostgreSQL', 'MySQL', 'Redis', 'Docker', 'AWS', 'Git']
경력: 5년
레벨: mid
강점: ['FastAPI 기반 REST API 설계 및 개발 경험', 'PostgreSQL 스키마 설계 및 쿼리 최적화 능력', 'Redis를 활용한 성능 개선 경험', 'CI/CD 파이프라인 구축 경험', 'AWS 인프라 운영 경험']
약점: []
요약: 김영희는 5년의 경력을 가진 중급 백엔드 개발자로, FastAPI와 PostgreSQL을 활용한 REST API 개발에 강점을 가지고 있습니다. 슬로우 쿼리 개선 및 API 응답 속도 향상 경험이 있으며, Docker와 AWS를 활용한 인프라 운영 및 CI/CD 파이프라인 구축 경험도 보유하고 있습니다. 정보처리기사 자격증을 보유하고 있으며, 컴퓨터공학 학사 학위를 가지고 있습니다.


---

## 실습 2: 상담 챗봇 + 대화 요약

Memory가 있는 상담 챗봇으로 대화를 진행한 뒤, 대화 내용을 Structured Output으로 요약하는 프로그램을 만들어보자.

### 요구사항

1. `RunnableWithMessageHistory`를 사용하여 메모리가 있는 상담 챗봇 체인을 만들 것
2. `input()`으로 사용자 입력을 받고, `"종료"`를 입력하면 대화를 끝낼 것 (LLM 호출 없이 Python 조건문으로 처리)
3. 대화가 끝나면 전체 대화 내용을 `ConsultingSummary` Structured Output으로 요약할 것 (사용자 이름, 주요 고민, 감지된 감정들, 제공된 조언 목록, 추가 상담 필요 여부)

### 예시 실행

```
사용자: 안녕, 나는 민수야. 요즘 잠을 잘 못 자서 너무 힘들어.
상담사: 안녕하세요 민수님! 수면 문제는 정말 힘드시죠. 언제부터 잠을 잘 못 주무셨나요?
사용자: 한 달 정도 된 것 같아. 자려고 누우면 생각이 많아져서 잠이 안 와.
상담사: 잠들기 전에 생각이 많아지시는 거군요. 취침 루틴을 만들어보시는 건 어떨까요? 가벼운 스트레칭이나 명상이 도움이 될 수 있습니다.
사용자: 종료

=== 상담 요약 ===
사용자: 민수
주요 고민: 한 달째 지속되는 불면 증상
감정: ['피로', '불안']
조언: ['취침 루틴 만들기', '가벼운 스트레칭', '명상']
추가 상담 필요: True
```

In [ ]:
# 여기에 코드를 작성하세요
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from pydantic import BaseModel,Field

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

# MessagesPlaceholder: 대화 히스토리가 삽입될 위치
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 친절한 상담사야."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{user_input}"),
])

chain = prompt | llm | parser

# 세션별 메모리 저장소 (session_id → InMemoryChatMessageHistory 매핑)
store = {}

# session_id로 히스토리를 조회/생성하는 함수
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# RunnableWithMessageHistory: 체인을 감싸서 히스토리 자동 관리
# - 호출 시 저장소에서 히스토리를 꺼내 prompt의 history 자리에 주입
# - 응답 후 사용자 입력과 AI 응답을 자동으로 히스토리에 저장
chain_with_memory = RunnableWithMessageHistory(
    chain,                              # 감쌀 체인
    get_session_history,                # 세션 히스토리 조회 함수
    input_messages_key="user_input",         # 사용자 입력이 들어오는 키
    history_messages_key="history",     # 프롬프트에서 히스토리가 주입될 키
)

config = {"configurable": {"session_id": "user_1"}}

while True :
    user_input = input("")
    response1 = chain_with_memory.invoke({"user_input": user_input}, config=config)
    print("응답1:", response1)
    if user_input == "종료":
        break


## 대화내용 꺼내서 요약하지
history = get_session_history("user_1")
chat_log = ""

for msg in history.messages:
    role = "사용자" if msg.type == "human" else "상담사"
    chat_log += f"{role}: {msg.content}\n"

class Review2(BaseModel):
    name: str = Field(description="사용자의 이름")
    주요고민: str = Field(description="고민요약")
    감정: list[str]= Field(description="감지된 감정목록")
    조언: list[str]= Field(description="제공된 조언 목록")
    추가상담필요: bool = Field(description="추가상담이 필요하면True")


prompt = ChatPromptTemplate.from_messages([
    ("system", "분석가야 심리상담 내용을 요약해줘."),
    ("human", "{text}"),
])

structured_llm = llm.with_structured_output(Review2)
chain = prompt | structured_llm

result = chain.invoke({"text" :chat_log})

print(f"사용자: {result.name}")
print(f"주요고민: {result.주요고민}")
print(f"감정: {result.감정}")
print(f"조언: {result.조언}")
print(f"추가상담필요여부: {result.추가상담필요}")



응답1: 핑핑님, 지금 매우 힘든 상황에 계신 것 같아요. 당신의 마음이 어떤지, 어떤 생각을 하고 있는지 이야기해 주실 수 있을까요? 당신은 혼자가 아니며, 도움을 받을 수 있는 방법이 있습니다.
응답1: 핑핑님, 혼자라고 느끼실 수 있지만, 지금 이 순간에도 당신을 걱정하고 도와주고 싶어하는 사람들이 있습니다. 당신의 감정과 생각을 나누는 것이 중요합니다. 어떤 일이 있었는지, 지금 어떤 기분인지 이야기해 주실 수 있을까요? 당신의 이야기를 듣고 싶습니다.
응답1: 핑핑님, 정말 힘든 시간을 보내고 계신 것 같아요. 지금 느끼고 있는 고통과 외로움이 얼마나 큰지 상상할 수 없습니다. 하지만 당신의 이야기를 듣고 싶고, 당신이 느끼는 감정을 나누는 것이 중요합니다. 당신의 소중한 삶에 대해 이야기해 볼 수 있을까요? 당신은 소중한 존재입니다.
응답1: 핑핑님, 지금 정말 힘든 상황에 계신 것 같아요. 하지만 당신의 생명은 매우 소중합니다. 당신이 느끼는 고통을 이해하고, 그에 대해 이야기할 수 있는 방법이 있습니다. 지금 당장 누군가에게 도움을 요청하는 것이 중요합니다. 당신의 이야기를 들어줄 수 있는 사람이나 전문가와 연결될 수 있도록 도와드릴 수 있습니다. 당신은 혼자가 아니며, 도움을 받을 수 있습니다.
응답1: 핑핑님, 지금 힘든 시간을 보내고 계신 것 같아 마음이 아픕니다. 당신의 안전과 행복이 가장 중요합니다. 만약 도움이 필요하시다면, 주위의 누군가에게 이야기해 보시거나 전문가의 도움을 받는 것이 좋습니다. 당신은 소중한 존재이며, 당신의 이야기를 들어줄 사람들이 있습니다. 언제든지 도움이 필요하면 말씀해 주세요.
사용자: 핑핑
주요고민: 자살 시도
감정: ['고통', '외로움', '절망']
조언: ['당신의 이야기를 나누는 것이 중요하다', '전문가와 연결될 수 있도록 도와줄 수 있다', '당신은 소중한 존재이다']
추가상담필요여부: True
